# 06 · From the research graphs to published results

**Goal.** Run the whole chain once, on the hosted PMDco 3 graphs: exploration, acquisition and analysis of all ten tests, the results written as RDF in the data graph's own vocabulary, the run recorded by semantikon, and both published with a publishing workflow, as an Ontodocker dataset and a Dataportal record. Every step is a workflow node; the publishing nodes wrap the praeco calls that [notebook 00](00-infrastructure-tour.ipynb) makes directly.

**Audience.** All readers; the shared end of every reading path.

**Prerequisites.**
- The steps of the chain as introduced on your path: the data and profiles ([01](01-tensile-data-from-graphs.ipynb)), the method ([02](02-tensile-method.ipynb)), the workflows ([03](03-tensile-workflow-building.ipynb)), the knowledge graphs ([04](04-tensile-workflow-knowledge-graph.ipynb)) or the results ([05](05-tensile-analysis-overview.ipynb)).
- The `pmd-demonstrator` environment; `ONTODOCKER_ADDRESS` (and `ONTODOCKER_TOKEN` if required) set before Jupyter starts, and for publishing `DATAPORTAL_TOKEN`. None of them is printed here.

**Reading paths.** MSE domain experts: [05](05-tensile-analysis-overview.ipynb) → [02](02-tensile-method.ipynb) → [01](01-tensile-data-from-graphs.ipynb) → **06**. Ontologists and RDF experts: [01](01-tensile-data-from-graphs.ipynb) → [04](04-tensile-workflow-knowledge-graph.ipynb) → [05](05-tensile-analysis-overview.ipynb) → **06**. Workflow developers: [03](03-tensile-workflow-building.ipynb) → [04](04-tensile-workflow-knowledge-graph.ipynb) → [02](02-tensile-method.ipynb) → [05](05-tensile-analysis-overview.ipynb) → **06**. Readers new to Ontodocker and the Dataportal may start with [00](00-infrastructure-tour.ipynb).

| Symbol | Meaning | Unit |
| --- | --- | --- |
| E | slope of the elastic line | GPa |
| ReH, ReL, Rp0.2, Rm | upper and lower yield strength, proof strength, tensile strength | MPa |
| Fm | maximum force | N |
| Ag, Agt, A, Z | extension at maximum force (plastic, total), elongation after fracture, reduction of area | % |

How each is computed and where the method has limits: [notebook 02](02-tensile-method.ipynb).

*Execution class: network-read* with the default `publish = False`: SPARQL queries and Turtle downloads from Ontodocker, curve downloads from the public data host, public reads of the Dataportal; files are written to a temporary directory only. With `publish = True` it is *remote-write*: it creates the Ontodocker dataset and the Dataportal record named below, which must not exist.

## Setup

The next cell is tagged `parameters` for papermill.

In [ ]:
publish = False
ontodocker_dataset = "pmdco3_tto_example_results"
record_name = "pmdco3-tto-results"

In [ ]:
import datetime
import hashlib
import importlib.metadata
import json
import tempfile
import warnings
from pathlib import Path

import pandas as pd
import pyiron_workflow as pwf
import rdflib
import semantikon
from IPython.display import display
from praeco.exceptions import HttpError

from pmd_tensile import knowledge_graph, orchestration, sources
from pmd_tensile.nodes import publishing
from pmd_tensile.nodes.results import MARKING, NAMESPACE

# semantikon's provenance hashing triggers an upstream networkx notice about changed hashes.
warnings.filterwarnings("ignore", message="The hashes produced for directed graphs", category=UserWarning)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.precision", 4)

PROFILE_ID = "pmdco3-tto"
DATASETS = [f"pmdco3_tto_example_{group}" for group in ("parallel", "perpendicular", "diagonal")]
SOURCE = {"kind": "ontodocker", "datasets": DATASETS}
workdir = tempfile.TemporaryDirectory(prefix="pmd-results-")  # removed in the last cell

## 1. The analysis, ending in the results node

`run_tensile_analysis` runs the exploration and the batch as in notebooks 03 to 05; with `results_graph=True` the batch ends in the results node `results_graph`, which writes the computed properties in the vocabulary of the PMDco 3 data graph.

In [ ]:
retrieved = pd.Timestamp.now(tz="Europe/Berlin").floor("s")
run = orchestration.run_tensile_analysis(PROFILE_ID, SOURCE, results_graph=True)
print("data retrieved:", retrieved)
print(f"batch run {run.batch_run.status}: {len(run.specimen_ids)} specimens; results graph: "
      f"{len(run.results_ntriples.splitlines())} triples")
display(run.results.set_index("specimen_id")[["orientation", "yield_type", "e_gpa", "reh_mpa", "rel_mpa", "rp02_mpa", "rm_mpa"]])

## 2. The results graph

Each specimen gets an *analytical calculation* (`pmdco:PMD_0025012`) whose input is the test's curve table, with one *calculation datum* (`pmdco:PMD_0025035`) per defined property. The datum is about the same quality individual of the material as the lab's *measurement datum* (`obo:IAO_0000109`), so one query puts both side by side; the two differ by class, by the process that produced them, and by the comment that marks every individual this demonstrator wrote. Undefined values are not written; flags are textual entities about the quality. The query, on the results graph merged with the hosted data graph of the parallel group:

In [ ]:
results_graph = rdflib.Graph().parse(data=run.results_ntriples, format="nt")
data_graph = rdflib.Graph().parse(data=sources.ontodocker_client().datasets.fetch_turtle("pmdco3_tto_example_parallel"),
                                  format="turtle")
SIDE_BY_SIDE = '''
PREFIX obo:   <http://purl.obolibrary.org/obo/>
PREFIX pmdco: <https://w3id.org/pmd/co/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?property ?computed ?computed_unit ?reported ?reported_unit WHERE {
    ?computed_datum a pmdco:PMD_0025035 ; obo:IAO_0000136 ?quality ; obo:OBI_0001938 ?computed_value .
    ?computed_value pmdco:PMD_0000006 ?computed ; obo:IAO_0000039 ?computed_unit .
    ?reported_datum a obo:IAO_0000109 ; obo:IAO_0000136 ?quality ; obo:OBI_0001938 ?reported_value .
    ?reported_value pmdco:PMD_0000006 ?reported .
    OPTIONAL { ?reported_value obo:IAO_0000039 ?reported_unit }
    FILTER(CONTAINS(STR(?computed_datum), "/Zx3/"))
    BIND(STRAFTER(STR(?computed_datum), "/Zx3/") AS ?property)
}
ORDER BY ?property
'''
print(SIDE_BY_SIDE.strip())
rows = (results_graph + data_graph).query(SIDE_BY_SIDE)
def unit(term):
    return str(term).rsplit("/", 1)[-1] if term is not None else "–"


display(pd.DataFrame([[str(r.property), float(r.computed), unit(r.computed_unit), float(r.reported), unit(r.reported_unit)]
                      for r in rows], columns=["property (Zx3)", "computed", "unit", "reported by the lab", "unit (lab)"]))
marked = {s for s in results_graph.subjects() if str(s).startswith(NAMESPACE)}
print(f"{len(marked)} individuals written, each marked: "
      f"{all((s, rdflib.RDFS.comment, rdflib.Literal(MARKING)) in results_graph for s in marked)}")

## 3. The run graph, and how the results are tied to it

semantikon exports the batch run (notebook 04) with every input and output value. The results graph cannot name the run graph's nodes, which semantikon creates only after the run; it records instead the SHA-256 of each curve file, which the run graph holds as a value. A query over both graphs joins them on that value:

In [ ]:
run_graph = semantikon.get_knowledge_graph(run.batch_run.result)
conforms, _ = knowledge_graph.validate_knowledge_graph(run_graph)
JOIN = '''
PREFIX obo:   <http://purl.obolibrary.org/obo/>
PREFIX pmdco: <https://w3id.org/pmd/co/>
SELECT ?calculation ?checksum (COUNT(DISTINCT ?run_node) AS ?run_nodes) WHERE {
    ?calculation a pmdco:PMD_0025012 ; obo:OBI_0000293 ?identifier .
    ?identifier a obo:IAO_0020000 ; pmdco:PMD_0000006 ?checksum .
    ?run_node pmdco:PMD_0000006 ?value .
    FILTER(?run_node != ?identifier && STR(?value) = STR(?checksum))
}
GROUP BY ?calculation ?checksum ORDER BY ?calculation
'''
joined = pd.DataFrame([[str(r.calculation).split("/")[-2], str(r.checksum)[:16] + "…", int(r.run_nodes)]
                       for r in (results_graph + run_graph).query(JOIN)],
                      columns=["specimen", "curve SHA-256", "run-graph data nodes with this value"])
print(f"run graph: {len(run_graph)} triples, conforms: {conforms}")
display(joined)

## 4. Publishing

The publishing workflow `publishing_workflow` writes the files to a directory, creates a new Ontodocker dataset with the results graph, and creates a Dataportal record from a reviewed description with all files as assets, adding the SHA-256 of each file to the record. It is composed of three nodes, each refusing to overwrite an existing file, dataset or record; tokens come from the environment. The files and the record:

In [ ]:
folder = Path(workdir.name)
run_graph_path = knowledge_graph.write_graph(run_graph, folder / "run.nt")  # N-Triples keeps every literal
contents = {
    "pmdco3_tto_results.nt": run.results_ntriples,
    "pmdco3_tto_results_run.nt": run_graph_path.read_text(encoding="utf-8"),
    "pmdco3_tto_results.csv": run.results.to_csv(index=False),
}
record = {
    "name": record_name,
    "title": "Computed tensile properties of ten S355 tests (PMDco 3)",
    "description": (
        "Tensile properties (E, ReH, ReL, Rp0.2, Fm, Rm, Ag, Agt, A, Z) computed by the PMD demonstrator's workflow "
        "(pmd_tensile, method version 3) from the raw curves of the ten S355 tests in the record "
        "pmdco3-tto-recombined, written in PMDco 3/TTO terms as calculation data next to the lab's measurement data. "
        "Includes the semantikon run graph of the computation. Inspired by ISO 6892-1; no claim of conformity."
    ),
    "creators": [{"name": "Marian Bruns", "orcid": "0000-0001-6476-7444"}],
    "license": "CC-BY-4.0",
    "keywords": ["Platform MaterialDigital", "PMD", "PMDco 3", "TTO", "S355", "tensile test", "RDF", "semantikon",
                 "pyiron_workflow"],
    "publication_date": datetime.date.today().isoformat(),
    "owner_org": "pmd",
    "private": False,
    "extras": {"pmd_profile": PROFILE_ID, "pmd_tensile": importlib.metadata.version("pmd-tensile"),
               "method": "method version 3", "source_datasets": json.dumps(DATASETS, separators=(",", ":"))},
}
display(pd.DataFrame({"file": list(contents), "size / kB": [round(len(t.encode()) / 1000, 1) for t in contents.values()]}))
display(pd.Series({k: v for k, v in record.items() if k != "extras"}, name="record").to_frame())

In [ ]:
if publish:
    (folder / "publication").mkdir()
    publication = pwf.node(publishing.publishing_workflow).run(
        contents=contents, directory=str(folder / "publication"), ontodocker_dataset=ontodocker_dataset,
        graph_file="pmdco3_tto_results.nt", record=record,
    )
    print("Ontodocker dataset:", publication.outputs.dataset, "| Dataportal record:", publication.outputs.record_name)
    publishing_graph = semantikon.get_knowledge_graph(publication.result)
else:
    print("preview only: nothing is written to a remote service")
    publishing_graph = semantikon.get_knowledge_graph(publishing.publishing_workflow.flowrep_recipe)
print(f"the publishing workflow's own graph: {len(publishing_graph)} triples, conforms: "
      f"{knowledge_graph.validate_knowledge_graph(publishing_graph)[0]}; it is shown here and not published")

## 5. What others get

A public, anonymous read of the record, and a SPARQL count on the new Ontodocker dataset (with `publish = False`, of what may exist from an earlier run):

In [ ]:
dataportal = sources.dataportal_client()
try:
    published = dataportal.datasets.show(record_name).raw
except HttpError as error:  # a public read: no record yet
    published = None
    print(f"{record_name}: no public record (status {error.status_code})")
if published:
    extras = {item["key"]: item["value"] for item in published.get("extras", [])}
    display(pd.DataFrame([{"asset": r["name"], "format": r["format"],
                           "SHA-256 as published = local file": extras.get(f"sha256:{r['name']}") ==
                           hashlib.sha256(contents[r["name"]].encode()).hexdigest()}
                          for r in published["resources"] if r["name"] in contents]))
    print("title:", published["title"], "| creators:", ", ".join(c["name"] for c in published.get("creator") or []))
ontodocker = sources.ontodocker_client()
if ontodocker_dataset in ontodocker.datasets.list():
    count = ontodocker.sparql.query_df(ontodocker_dataset, "SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }", ["n"])
    print(f"{ontodocker_dataset}: {int(count['n'][0])} triples hosted; results graph: {len(results_graph)}")

## Summary

- One workflow run analysed the ten tests from the hosted PMDco 3 graphs and ended in the results node (section 1).
- The results graph puts each computed value next to the lab's on the same quality, marked as the demonstrator's; the lab's graph uses its own unit IRIs (e.g. kN for Fm), the results graph QUDT's (section 2).
- The run graph conforms and is joined with the results by the curve checksums (section 3).
- The publishing workflow published the results graph on Ontodocker and all files as a Dataportal record; its own graph is shown, not published (sections 4 and 5).

In [ ]:
workdir.cleanup()